# Taux de complétion (via API)

Utilisation de l'API aggrégé de matomo pour récupérer les taux de complétion des outils

## Taux de completion par simulateur

Taux de complétion par simulateur = sur 100 visites sur nos simulateurs, x ont été jusqu'au résultat (idéalement j'aimerais avoir l'info pour chaque simulateur puisque les parcours varient d'un simulateur à l'autre).

In [ ]:
import requests
import pandas as pd
import re

In [ ]:
base_url = "https://matomo.fabrique.social.gouv.fr/index.php"
period = "day" # range / day
date = "2026-06-01" # 2026-06-01,2026-06-30

In [ ]:
from analysis.reports.completion_simulateurs import get_completion_simulateurs

df_test = get_completion_simulateurs(date)
df_test

## Taux de completion des contributions

Taux de lecture des contributions = sur 100 visites qui arrivent sur nos pages contribution, x ont effectivement accédé au contenu (générique ou personnalisé d'après la convention collective).

In [ ]:
base_slugs = [
    "quel-est-le-salaire-minimum",
    "quel-est-le-salaire-minimum-dun-alternant-en-2026",
    "le-preavis-de-licenciement-doit-il-etre-execute-en-totalite-y-compris-si-le-salarie-a-retrouve-un-emploi",
    "quelle-est-la-duree-du-conge-de-maternite",
    "dans-le-cadre-dun-cdd-quel-est-le-montant-de-lindemnite-de-fin-de-contrat",
    "quelle-est-la-duree-maximale-du-contrat-de-mission-interim",
    "quelle-est-la-duree-de-preavis-en-cas-de-depart-a-la-retraite",
    "si-le-salarie-est-malade-pendant-ses-conges-quelles-en-sont-les-consequences",
    "si-un-poste-se-libere-ou-est-cree-dans-lentreprise-lemployeur-doit-il-en-informer-les-salaries-ou-le-leur-proposer-en-priorite",
    "heures-supplementaires",
    "quelles-sont-les-conditions-de-cumul-demplois",
    "conges-supplementaires-pour-anciennete",
    "faut-il-respecter-un-delai-de-carence-entre-deux-cdd-si-oui-quelle-est-sa-duree",
    "faut-il-respecter-un-delai-de-carence-entre-deux-contrats-de-mission-interim",
    "le-salarie-peut-il-sabsenter-pour-rechercher-un-emploi-pendant-son-preavis",
    "les-conges-pour-evenements-familiaux",
    "lentreprise-peut-elle-embaucher-dans-le-cadre-dun-cdi-de-chantier-ou-doperation",
    "quelle-peut-etre-la-duree-maximale-dun-cdd",
    "comment-determiner-lanciennete-du-salarie",
    "a-quelles-indemnites-peut-pretendre-un-salarie-qui-part-a-la-retraite",
    "embauche-en-contrat-dextra-cdd-dusage",
    "quelles-informations-doivent-figurer-dans-le-contrat-de-travail-ou-la-lettre-dengagement",
    "quelles-sont-les-conditions-dattribution-de-la-prime-pour-travaux-dangereux-et-de-la-prime-pour-travaux-insalubres",
    "en-cas-de-perte-de-marche-par-lemployeur-quelles-sont-les-conditions-dun-transfert-des-contrats-de-travail",
    "arret-maladie-pendant-la-periode-dessai-quelles-sont-les-regles",
    "quest-ce-quune-rupture-conventionnelle",
    "combien-de-fois-le-contrat-de-travail-peut-il-etre-renouvele",
    "arret-maladie-pendant-le-preavis-quelles-consequences",
    "quelles-sont-les-primes-prevues-par-la-convention-collective",
    "quand-le-salarie-a-t-il-droit-a-une-prime-danciennete-quel-est-son-montant",
    "dans-le-cadre-dun-contrat-de-mission-interim-quel-est-le-montant-de-lindemnite-de-fin-de-contrat",
    "quelles-sont-les-consequences-du-non-respect-du-preavis-par-le-salarie-ou-lemployeur",
    "quelle-est-la-duree-maximale-de-la-periode-dessai-sans-et-avec-renouvellement",
    "la-periode-dessai-peut-elle-etre-renouvelee",
    "quelles-sont-les-conditions-de-la-clause-de-non-concurrence",
    "quelle-est-la-duree-de-preavis-en-cas-de-mise-a-la-retraite",
    "travail-du-dimanche-quelle-contrepartie",
    "quelle-est-la-duree-du-preavis-en-cas-de-demission",
    "en-cas-de-maladie-le-salarie-a-t-il-droit-a-une-garantie-demploi",
    "quelles-sont-les-consequences-du-deces-de-lemployeur-sur-le-contrat-de-travail",
    "jours-feries-et-ponts-dans-le-secteur-prive",
    "quelle-est-la-duree-de-preavis-en-cas-de-licenciement",
    "est-il-obligatoire-davoir-un-contrat-de-travail-ecrit-et-signe",
    "le-preavis-de-demission-doit-il-etre-execute-en-totalite-y-compris-si-le-salarie-a-retrouve-un-emploi",
    "en-cas-darret-maladie-du-salarie-lemployeur-doit-il-assurer-le-maintien-de-salaire",
    "quelles-sont-les-conditions-dindemnisation-pendant-le-conge-de-maternite",
]

In [ ]:
import time

BASE = "https://code.travail.gouv.fr/contribution/"

def fetch_visits_segment(slug):
    segment = (
        f"pageUrl=={BASE}{slug},"      # exact
        f"pageUrl=^{BASE}{slug}/,"     # suivi de /
        f"pageUrl=^{BASE}{slug}?,"     # suivi de ?
        f"pageUrl=^{BASE}{slug}#"      # suivi de #
    )
    params = {
        "module": "API",
        "method": "VisitsSummary.get",
        "idSite": 4,
        "period": "day",
        "date": date,
        "format": "JSON",
        "segment": segment,   # requests s'occupe de l'URL-encodage
    }
    resp = requests.get(base_url, params=params)
    resp.raise_for_status()
    data = resp.json()
    return int(data.get("nb_visits", 0))

rows = []
for slug in base_slugs:
    rows.append({"slug": slug, "nb_visites_uniques": fetch_visits_segment(slug)})
    time.sleep(0.2)   # soulage le serveur si archivage à la volée

result_seg = (
    pd.DataFrame(rows)
    .sort_values("nb_visites_uniques", ascending=False)
    .reset_index(drop=True)
)
total = pd.DataFrame({
    "slug": ["TOTAL"],
    "nb_visites_uniques": [result_seg["nb_visites_uniques"].sum()],
})
result_seg = pd.concat([result_seg, total], ignore_index=True)

In [ ]:
def fetch_event_visits(event_action):
    params = {
        "module": "API", "method": "Events.getAction", "idSite": 4,
        "period": "day", "date": date, "format": "JSON",
        "secondaryDimension": "eventName", "flat": 1, "filter_limit": -1,
        "filter_pattern": event_action,
    }
    resp = requests.get(base_url, params=params)
    resp.raise_for_status()
    df = pd.json_normalize(resp.json())
    df = df[df['Events_EventAction'] == event_action].copy()
    df['slug'] = df['Events_EventName'].str.extract(r"contribution/([^/?#]+)")[0]
    return df.groupby('slug')['nb_visits'].sum()

sans_cc = fetch_event_visits('click_afficher_les_informations_sans_CC')
avec_cc = fetch_event_visits('click_afficher_les_informations_CC')

In [ ]:
seg = result_seg[result_seg['slug'] != 'TOTAL'].copy()

seg['nb_visits_click_sans_CC'] = seg['slug'].map(sans_cc).fillna(0).astype(int)
seg['nb_visits_click_avec_CC'] = seg['slug'].map(avec_cc).fillna(0).astype(int)
seg['nb_visits_click_total'] = seg['nb_visits_click_sans_CC'] + seg['nb_visits_click_avec_CC']

# ratio, en évitant la division par zéro
seg['taux_click_%'] = (
    100 * seg['nb_visits_click_total'] / seg['nb_visites_uniques'].replace(0, pd.NA)
).round(1)

total = pd.DataFrame({
    'slug': ['TOTAL'],
    'nb_visites_uniques': [seg['nb_visites_uniques'].sum()],
    'nb_visits_click_sans_CC': [seg['nb_visits_click_sans_CC'].sum()],
    'nb_visits_click_avec_CC': [seg['nb_visits_click_avec_CC'].sum()],
    'nb_visits_click_total': [seg['nb_visits_click_total'].sum()],
})
total['taux_click_%'] = (
    100 * total['nb_visits_click_total'] / total['nb_visites_uniques'].replace(0, pd.NA)
).round(1)

result_full = pd.concat([seg, total], ignore_index=True)

In [ ]:
result_full

Version avec la dimension desktop / mobile

In [ ]:
import time
BASE = "https://code.travail.gouv.fr/contribution/"

segments = {
    "desktop": "deviceType==desktop",
    "mobile": "deviceType==smartphone",
}


def fetch_visits_segment(slug, device_segment=None):
    clauses = [
        f"pageUrl=={BASE}{slug}",      # exact
        f"pageUrl=^{BASE}{slug}/",     # suivi de /
        f"pageUrl=^{BASE}{slug}?",     # suivi de ?
        f"pageUrl=^{BASE}{slug}#",     # suivi de #
    ]
    # On applique le filtre device à CHAQUE clause pour garder
    # (A ou B ou C ou D) ET device, puisque ; est prioritaire sur ,
    if device_segment:
        clauses = [f"{c};{device_segment}" for c in clauses]
    segment = ",".join(clauses)

    params = {
        "module": "API",
        "method": "VisitsSummary.get",
        "idSite": 4,
        "period": "day",
        "date": date,
        "format": "JSON",
        "segment": segment,
    }
    resp = requests.get(base_url, params=params)
    resp.raise_for_status()
    return int(resp.json().get("nb_visits", 0))


def fetch_event_visits(event_action, device_segment=None):
    params = {
        "module": "API", "method": "Events.getAction", "idSite": 4,
        "period": "day", "date": date, "format": "JSON",
        "secondaryDimension": "eventName", "flat": 1, "filter_limit": -1,
        "filter_pattern": event_action,
    }
    if device_segment:
        params["segment"] = device_segment
    resp = requests.get(base_url, params=params)
    resp.raise_for_status()
    df = pd.json_normalize(resp.json())
    df = df[df['Events_EventAction'] == event_action].copy()
    df['slug'] = df['Events_EventName'].str.extract(r"contribution/([^/?#]+)")[0]
    return df.groupby('slug')['nb_visits'].sum()


def build_result_for_device(device_segment):
    # Visites uniques par slug
    rows = []
    for slug in base_slugs:
        rows.append({"slug": slug,
                     "nb_visites_uniques": fetch_visits_segment(slug, device_segment)})
        time.sleep(0.2)
    seg = (
        pd.DataFrame(rows)
        .sort_values("nb_visites_uniques", ascending=False)
        .reset_index(drop=True)
    )

    # Events clics avec / sans CC
    sans_cc = fetch_event_visits('click_afficher_les_informations_sans_CC', device_segment)
    avec_cc = fetch_event_visits('click_afficher_les_informations_CC', device_segment)

    seg['nb_visits_click_sans_CC'] = seg['slug'].map(sans_cc).fillna(0).astype(int)
    seg['nb_visits_click_avec_CC'] = seg['slug'].map(avec_cc).fillna(0).astype(int)
    seg['nb_visits_click_total'] = seg['nb_visits_click_sans_CC'] + seg['nb_visits_click_avec_CC']

    # Ligne TOTAL du device
    total = pd.DataFrame({
        'slug': ['TOTAL'],
        'nb_visites_uniques': [seg['nb_visites_uniques'].sum()],
        'nb_visits_click_sans_CC': [seg['nb_visits_click_sans_CC'].sum()],
        'nb_visits_click_avec_CC': [seg['nb_visits_click_avec_CC'].sum()],
        'nb_visits_click_total': [seg['nb_visits_click_total'].sum()],
    })

    return pd.concat([seg, total], ignore_index=True)


# Boucle sur les devices
parts = []
for device, device_segment in segments.items():
    part = build_result_for_device(device_segment)
    part.insert(0, "device", device)
    parts.append(part)

result_full = pd.concat(parts, ignore_index=True)
result_full

In [ ]:
from analysis.reports.contrib_cc_clicks import get_contrib_cc_clicks
df_contribs = get_contrib_cc_clicks(date)
df_contribs

## Taux de téléchargement des modèles

taux de téléchargement des modèles = part des visites sur une page modèle ayant donné lieu à un téléchargement ou un événement de copier/coller détecté. Attention : ce taux structurellement sous-estimé car le copier-coller manuel ne sera pas mesuré... Est-ce qu'on le mesure quand même ?

In [ ]:
from urllib.parse import urlparse, unquote

MODELES = [
    ("recu-pour-solde-de-tout-compte", "recu_pour_solde_de_tout_compte.docx"),
    ("document-dinformation-sur-le-conge-de-reclassement", "document-d-information-sur-le-conge-de-reclassement.docx"),
    ("signalement-de-faits-pouvant-relever-du-harcelement-moral-ou-sexuel", "signalement_harcelement.docx"),
    ("affichage-obligatoire-relatif-au-harcelement-sexuel", "affichage_lutte_contre_harcelement_sexuel.docx"),
    ("informations-principales-relatives-a-la-relation-de-travail-delivrees-au-salarie", "Modèle de document regroupant les quatorze informations principales relatives à la relation de travail délivrées au salarié.docx"),
    ("lettre-de-licenciement-economique-sans-entretien-prealable-conge-de-reclassement", "Lettre-de-licenciement-economique-sans-entretien-prealable-conge-de-reclassement.docx"),
    ("lettre-de-licenciement-suite-a-un-accord-de-performance-collective-apc", "lettre-de-licenciement-apc.docx"),
    ("demande-de-paiement-de-salaire", "demande_paiement_salaire.docx"),
    ("lettre-de-licenciement-pour-inaptitude", "lettre-de-licenciement-pour-inaptitude.docx"),
    ("lettre-de-licenciement-economique-avec-entretien-prealable-conge-de-reclassement", "Lettre-de-licenciement-economique-avec-entretien-prealable-conge-de-reclassement.docx"),
    ("rupture-de-periode-dessai-par-lemployeur", "rupture_periode_d-essai_employeur.docx"),
    ("reponse-a-un-signalement-de-harcelement-sexuel", "reponse_a_signalement_harcelement_sexuel.docx"),
    ("reclamation-de-conges-payes", "reclamation_de_cp.docx"),
    ("lettre-de-licenciement-economique-envoyee-a-titre-conservatoire-csp", "Lettre_de_licenciement_economique_envoyee_a_titre_conservatoire _CSP.docx"),
    ("notification-du-depart-du-salarie-a-la-retraite", "Notification_depart_retraite.docx"),
    ("demande-a-lemployeur-dorganisation-des-visites-medicales", "Demande_organisation_des_visites_medicales.docx"),
    ("certificat-de-travail", "certificat_de_travail.docx"),
    ("convocation-a-un-entretien-prealable-au-licenciement-pour-motif-personnel", "entretien_prealable_au_licenciement_personnel.docx"),
    ("conge-supplementaire-de-naissance-comment-informer-son-employeur", "modèle congé de naissance.docx"),
    ("attestation-de-travail", "Attestation.de.travail.docx"),
    ("demande-de-reprise-de-salaire-suite-a-une-declaration-dinaptitude", "reprise_de_salaire_inaptitude.docx"),
    ("proposition-dentretien-en-vue-dune-eventuelle-rupture-conventionnelle", "Proposition.d.entretien.en.vue.d.une.eventuelle.rupture.conventionnelle.docx"),
    ("rupture-anticipee-du-cdd-suite-a-une-embauche-en-cdi", "rupture_anticipee_du_CDD_suite_a_une_embauche_en_CDI.docx"),
    ("lettre-de-demission", "lettre_de_demission.docx"),
    ("signalement-de-harcelement-sexuel", "signalement_harcelement_sexuel.docx"),
    ("priorite-demploi-lettre-permettant-a-lemployeur-de-porter-a-la-connaissance-des-salaries-concernes-les-emplois-disponibles", "Priorite.d.emploi-lettre.permettant.a.l.employeur.de.porter.a.la.connaissance.des.salaries.concernes.les.emplois.disponibles.docx"),
    ("rupture-dun-commun-accord-conge-de-mobilite", "Rupture-d-un-commun-accord-conge-de-mobilite.docx"),
    ("rupture-dun-commun-accord-rupture-conventionnelle-collective", "Rupture-d-un-commun-accord-rupture-conventionnelle-collective.docx"),
    ("lettre-de-licenciement-economique-envoyee-a-titre-definitif-csp", "Lettre_de_licenciement_economique_envoyee_a_titre_definitif_csp.docx"),
    ("demande-de-rendez-vous-en-vue-dune-rupture-conventionnelle", "demande_rdv_rupture_conventionnelle.docx"),
    ("convocation-a-un-entretien-prealable-a-une-sanction-disciplinaire", "convocation_entretien_prealable_sanction_disciplinaire.docx"),
    ("contrat-de-travail-a-duree-determinee-cdd", "Contrat_de_travail_à_durée_déterminée.docx"),
    ("lettre-accusant-reception-dune-lettre-de-demission", "accuse_reception_lettre_de_demission.docx"),
    ("lettre-de-reclamation-des-documents-de-fin-de-contrat", "reclamation_des_documents_de_fin_de_contrat.docx"),
    ("modele-de-lettre-de-prise-de-conge-paternite-et-daccueil", "lettre_de_congé_de_paternité_et_d'acceuil_d'enfant.docx"),
    ("lettre-de-licenciement-pour-faute-simple-ou-grave-du-salarie-du-particulier-employeur", "lettre_licenciement_faute_simple_ou_grave_salarié_particulier_employeur.docx"),
    ("contrat-de-travail-a-duree-indeterminee", "Contrat_de_travail_à_durée_indéterminée.docx"),
    ("licenciement-economique-lettre-de-rupture-en-cas-dadhesion-a-un-csp", "Licenciement.economique.-.lettre.de.rupture.en.cas.d.adhesion.a.un.CSP.docx"),
    ("rupture-du-contrat-en-periode-dessai-par-le-salarie", "rupture_periode_d-essai_salarie.docx"),
    ("promesse-dembauche", "Promesse _d_embauche.docx"),
    ("convocation-a-un-entretien-prealable-au-licenciement-du-salarie-du-particulier-employeur", "convocation_entretien_préalable_salarié_particulier_employeur.docx"),
    ("demande-daccord-du-salarie-pour-le-renouvellement-dune-periode-dessai", "renouvellement_periode_essai_initiative_employeur.docx"),
    ("lettre-de-licenciement-pour-motif-personnel-non-disciplinaire", "lettre_de_licenciement_pour_motif_non_disciplinaire.docx"),
    ("modele-de-mise-en-demeure-pour-abandon-de-poste", "lettre_de_mise_en_demeure_pour_abandon_de_poste.docx"),
    ("rupture-dun-contrat-dapprentissage-dun-commun-accord", "rupture_amiable_du_contrat_d_apprentissage.docx"),
    ("rupture-dun-contrat-de-travail-a-duree-determinee-dun-commun-accord", "rupture_commun_accord_du_cdd.docx"),
    ("demande-de-maintien-de-salaire-en-cas-darret-maladie", "Demande de maintien de salaire en cas d’arrêt maladie.docx"),
    ("informations-principales-relatives-a-la-relation-de-travail-du-salarie-appele-a-travailler-a-letranger", "Informations principales relatives à la relation de travail du salarié appelé à travailler à l'étranger.docx"),
    ("informations-principales-relatives-a-la-relation-de-travail-du-salarie-detache", "Informations principales relatives à la relation de travail d'un salarié détaché.docx"),
    ("lettre-de-licenciement-pour-motif-disciplinaire", "lettre_de_licenciement_pour_motif_disciplinaire.docx"),
    ("convocation-a-un-entretien-prealable-au-licenciement-economique-dau-moins-10-salaries-pendant-30-jours", "Entretien-prealable-au-licenciement-economique-d-au-moins-10-salaries-sur-30-jours.docx"),
    ("convocation-a-un-entretien-prealable-au-licenciement-economique-de-moins-de-10-salaries-pendant-30-jours", "entretien_prealable_au_licenciement_economique_de_moins_de_10_salaries_pendant_30_jours.docx"),
    ("demande-de-derogation-a-la-duree-minimale-de-travail-pour-un-temps-partiel", "temps_partiel_demande_de_derogation_du_salarie.docx"),
    ("releve-dheures-supplementaires", "tableau_recap_des_heures_supplementaires.docx"),
    ("lettre-de-reclamation-des-heures-supplementaires", "reclamation_des_heures_sup.docx"),
    ("demande-de-prolongation-et-ou-de-transformation-du-conge-parental-deducation-a-temps-plein", "Demande de Prolongation et ou transformation du congé parental à temps plein.docx"),
    ("demande-de-prolongation-et-ou-de-transformation-du-conge-parental-deducation-a-temps-partiel", "Demande de prolongation et ou de transformation du congé parental à temps partiel.docx"),
    ("demande-de-retour-anticipe-a-la-suite-dun-conge-parental-deducation-a-temps-plein", "Demande de retour anticipé à la suite d’un congé parental d’éducation à temps plein.docx"),
    ("demande-de-retour-anticipe-a-la-suite-dun-conge-parental-deducation-a-temps-partiel", "Demande de retour anticipé congé parental à temps partiel.docx"),
]

mapping_df = pd.DataFrame(MODELES, columns=["slug", "filename"])
mapping_df["filename_norm"] = mapping_df["filename"].map(lambda n: unquote(str(n)).strip().lower())
mapping_df["slug_lower"] = mapping_df["slug"].str.lower()

In [ ]:
def matomo_api(method, **extra):
    params = {
        "module": "API", "method": method,
        "idSite": 4, "period": "day", "date": date,
        "format": "JSON", "flat": 1,
        "filter_limit": -1,   # tout récupérer
    }
    params.update(extra)
    resp = requests.get(base_url, params=params)
    resp.raise_for_status()
    return pd.json_normalize(resp.json())

# --- Downloads ---
downloads_api = matomo_api("Actions.getDownloads")


In [ ]:
downloads_api

In [ ]:
def normalize(name) -> str:
    # décode + minuscule + trim pour comparer proprement
    return unquote(str(name)).strip().lower()

# NB: le champ contenant l'URL complète est en général 'url' en mode flat.
# Vérifie avec downloads_api.columns et adapte si besoin.
def extract_filename(url):
    if not isinstance(url, str): return ""
    return normalize(urlparse(url).path.rsplit("/", 1)[-1])

downloads_api["filename_norm"] = downloads_api["url"].map(extract_filename)
downloads_api = downloads_api.merge(
    mapping_df[["slug", "filename_norm"]], on="filename_norm", how="left"
)
dl_per_model = (
    downloads_api.dropna(subset=["slug"])
    .assign(nb_visits=lambda d: pd.to_numeric(d["nb_visits"], errors="coerce"))
    .groupby("slug")["nb_visits"].sum()
    .rename("nb_downloads_visits").reset_index()
)

In [ ]:
dl_per_model

In [ ]:
def fetch_event_modeles(event_action, device_segment=None):
    params = {
        "module": "API", "method": "Events.getAction", "idSite": 4,
        "period": "day", "date": date, "format": "JSON",
        "secondaryDimension": "eventName", "flat": 1, "filter_limit": -1,
        "filter_pattern": event_action,
    }
    if device_segment:
        params["segment"] = device_segment
    resp = requests.get(base_url, params=params)
    resp.raise_for_status()
    df = pd.json_normalize(resp.json())
    print(df.columns.tolist())
    df.head()
    df = df[df['Events_EventAction'] == event_action].copy()
    df['slug'] = df['Events_EventName'].str.extract(r"modeles-de-courriers/([^/?#]+)")[0]
    return df.groupby('slug')['nb_visits'].sum()

Ctrl_C_events = fetch_event_modeles('type_CTRL_C')
Ctrl_C_events

In [ ]:
import sys
import time

BASE = "https://code.travail.gouv.fr/modeles-de-courriers/"

# --- Vues de page ---
def fetch_visits_segment(slug):
    segment = (
        f"pageUrl=={BASE}{slug},"      # exact
        f"pageUrl=^{BASE}{slug}/,"     # suivi de /
        f"pageUrl=^{BASE}{slug}?,"     # suivi de ?
        f"pageUrl=^{BASE}{slug}#"      # suivi de #
    )
    params = {
        "module": "API",
        "method": "VisitsSummary.get",
        "idSite": 4,
        "period": "day",
        "date": date,
        "format": "JSON",
        "segment": segment,
    }
    resp = requests.get(base_url, params=params)
    resp.raise_for_status()
    return int(resp.json().get("nb_visits", 0))

ows = []
total = len(mapping_df)
t0 = time.time()

for i, slug in enumerate(mapping_df["slug"], start=1):
    try:
        nb = fetch_visits_segment(slug)
        status = f"{nb} visites"
    except Exception as e:
        nb = None
        status = f"ERREUR: {e}"

    elapsed = time.time() - t0
    print(f"[{i}/{total}] ({elapsed:5.1f}s) {BASE}{slug} -> {status}")
    sys.stdout.flush()   # force l'affichage immédiat dans le notebook

    rows.append({"slug": slug, "nb_pageviews_visits": nb})

pv_per_model = pd.DataFrame(rows)

# Récap des échecs éventuels
n_err = pv_per_model["nb_pageviews_visits"].isna().sum()
print(f"\nTerminé en {time.time() - t0:.1f}s — {total - n_err}/{total} OK, {n_err} en erreur")

In [ ]:
pv_per_model

In [ ]:
# --- Taux par modèle (downloads / visites) ---
taux_api = (
    pv_per_model.merge(dl_per_model, on="slug", how="left")
    .assign(nb_downloads_visits=lambda d: d["nb_downloads_visits"].fillna(0).astype(int))
)
taux_api["taux_download"] = (
    taux_api["nb_downloads_visits"] / taux_api["nb_pageviews_visits"].replace(0, pd.NA)
)
taux_api = taux_api.sort_values("taux_download", ascending=False)
taux_api